In [1]:
import os
os.environ["USE_TF"] = "0"

In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np, torch

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cpu


In [4]:
ds = load_dataset("imdb")

train = ds["train"].shuffle(seed=1).select(range(1500))
test = ds["test"].shuffle(seed=1).select(range(500))

In [5]:
name = "distilbert-base-uncased"

tok = AutoTokenizer.from_pretrained(name)

model = AutoModelForSequenceClassification.from_pretrained(
    name, num_labels=2
).to(device)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [6]:
def token(b):
    return tok(
        b["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

In [7]:
train = train.map(token, batched=True)
test = test.map(token, batched=True)

train.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "label"]
)

test.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "label"]
)

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [8]:
def metrics(p):
    x = np.argmax(p[0], axis=1)
    return {"accuracy": (x == p[1]).mean()}

args = TrainingArguments(
    output_dir="out",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="no",
    report_to=[]
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train,
    eval_dataset=test,
    compute_metrics=metrics
)

trainer.train()
print(trainer.evaluate())

C:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
text = "This movie was fantastic!"
x = tok(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=128
).to(device)

pred = torch.argmax(
    model(**x).logits,
    dim=1
).item()